# Predicting Bank Note Authentication

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

In [7]:
import pandas as pd

def clean_treasury_data(file_path, output_path):
    print("🚀 Initializing Treasury Risk ETL Pipeline...")

    # 1. Ingest Raw Geometric Data (Using semicolon separator)
    df = pd.read_csv(file_path, sep=';')

    # 2. Check for missing values
    null_counts = df.isnull().sum()
    print(f"📊 Missing values detected per column:\n{null_counts}\n")

    # 3. Smart Imputation: Fill missing low margin values using the median of its specific class
    # This ensures a fake bill's missing margin gets a fake bill's average margin size.
    df['margin_low'] = df.groupby('is_genuine')['margin_low'].transform(lambda x: x.fillna(x.median()))

    # 4. Standardize Target Labels for Power BI (True/False -> Genuine/Counterfeit)
    df['Status'] = df['is_genuine'].map({True: 'Genuine', False: 'Counterfeit'})
    df.drop(columns=['is_genuine'], inplace=True)

    # 5. Export Production Asset
    df.to_csv(output_path, index=False)
    print(f"📦 Production data exported successfully to: {output_path}")

# Run the pipeline
clean_treasury_data('fake_bills (1).csv', 'Clean_Treasury_Bills.csv')

🚀 Initializing Treasury Risk ETL Pipeline...
📊 Missing values detected per column:
is_genuine       0
diagonal         0
height_left      0
height_right     0
margin_low      37
margin_up        0
length           0
dtype: int64

📦 Production data exported successfully to: Clean_Treasury_Bills.csv


# Importing the Data

In [2]:
from google.colab import files
import io
import pandas as pd

# Upload file
print("Please upload your fake_bills.csv file:")
uploaded = files.upload()

# Get filename
filename = list(uploaded.keys())[0]
print(f"Loading file: {filename}")

# READ WITH SEMICOLON SEPARATOR
df = pd.read_csv(io.BytesIO(uploaded[filename]), sep=';')

# Now check the data
print("\n✅ First 5 rows:")
print(df.head())

print("\n✅ Column names:")
print(df.columns.tolist())

print("\n✅ Missing values:")
print(df.isna().sum())

Please upload your fake_bills.csv file:


Saving fake_bills.csv to fake_bills (1).csv
Loading file: fake_bills (1).csv

✅ First 5 rows:
   is_genuine  diagonal  height_left  height_right  margin_low  margin_up  \
0        True    171.81       104.86        104.95        4.52       2.89   
1        True    171.46       103.36        103.66        3.77       2.99   
2        True    172.69       104.48        103.50        4.40       2.94   
3        True    171.36       103.91        103.94        3.62       3.01   
4        True    171.73       104.28        103.46        4.04       3.48   

   length  
0  112.83  
1  113.09  
2  113.16  
3  113.51  
4  112.54  

✅ Column names:
['is_genuine', 'diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up', 'length']

✅ Missing values:
is_genuine       0
diagonal         0
height_left      0
height_right     0
margin_low      37
margin_up        0
length           0
dtype: int64


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   is_genuine    1500 non-null   bool   
 1   diagonal      1500 non-null   float64
 2   height_left   1500 non-null   float64
 3   height_right  1500 non-null   float64
 4   margin_low    1463 non-null   float64
 5   margin_up     1500 non-null   float64
 6   length        1500 non-null   float64
dtypes: bool(1), float64(6)
memory usage: 71.9 KB


# Checking the Preliminary Information

* There is no missing values in the dataset

In [4]:
# Fill missing values with median
for col in ['margin_low', 'margin_up']:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)
    print(f"Filled {col} with median: {median_val:.2f}")

# Verify no more missing values
print("\n✅ Missing values after filling:")
print(df.isna().sum())

Filled margin_low with median: 4.31
Filled margin_up with median: 3.14

✅ Missing values after filling:
is_genuine      0
diagonal        0
height_left     0
height_right    0
margin_low      0
margin_up       0
length          0
dtype: int64


# Getting the Statistical Information about continouous features

In [5]:
df.describe()

,diagonal,height_left,height_right,margin_low,margin_up,length
count,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.00000
mean,171.958440,104.029533,103.920307,4.481627,3.151473,112.67850
std,0.305195,0.299462,0.325627,0.656137,0.231813,0.87273
min,171.040000,103.140000,102.820000,2.980000,2.270000,109.49000
25%,171.750000,103.820000,103.710000,4.030000,2.990000,112.03000
50%,171.960000,104.040000,103.920000,4.310000,3.140000,112.96000
75%,172.170000,104.230000,104.150000,4.860000,3.310000,113.34000
max,173.010000,104.880000,104.950000,6.900000,3.910000,114.44000


# Class distribution of Target column

In [6]:
df['class'].value_counts().plot(kind='bar')
plt.title('Class Distribution for Target Column')
plt.ylabel('count')
plt.xlabel('Class')
plt.show()

KeyError: 'class'

# Univariant Analysis with Histogram

In [ ]:
df.hist(bins=20,figsize=(11,9),layout=(2,3))
plt.show()

# PairPlot for Target Column with respect to Features Columns

In [ ]:
sns.pairplot(df,hue='class')
plt.show()

# Checking for Correlation Matrix

In [ ]:
corr=df.corr()
sns.heatmap(corr,annot=True,cmap='mako')
plt.title('Correlation Matrix between Features')
plt.show()

# Preprocessing Function

In [ ]:
def preprocess_inputs(df):
    df=df.copy()
    #splitting the data between target and feature dataset
    y=df['class']
    x=df.drop('class',axis=1)
    #train_test_split
    x_train,x_test,y_train,y_test=train_test_split(x,y,train_size=0.7)
    #scaling the dataset

    scaler=StandardScaler()
    scaler.fit(x_train)
    x_train=pd.DataFrame(scaler.transform(x_train),columns=x_train.columns)
    x_test=pd.DataFrame(scaler.transform(x_test),columns=x_test.columns)

    return x_train,x_test,y_train,y_test





In [ ]:
x_train,x_test,y_train,y_test=preprocess_inputs(df)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

# Randomn Forest Classifier

In [ ]:
model=RandomForestClassifier()
model.fit(x_train,y_train)
print(model.score(x_test,y_test))

# Checking the Accuracy with Confusion Matrix, and Classification Report

In [ ]:
y_pred=model.predict(x_test)
cm=confusion_matrix(y_test,y_pred)
sns.heatmap(cm,annot=True,cmap='mako')
plt.show()

# Classication Report

In [ ]:
clr=classification_report(y_pred,y_test)
print(clr)